In [ ]:
# Imports and Configuration
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
# ---- EDIT THIS PATH ----
INPUT_PATH = "C:\\Users\\USER\\Desktop\\Research_My_RAG\\dataset\\miriad_cardiology.jsonl"
# ------------------------

In [4]:
OUTPUT_TRAIN = "t5_hyde_train_split.jsonl"
OUTPUT_VAL = "t5_hyde_val_split.jsonl"

In [5]:
# Load, Filter, and Balance Data
print("Loading raw dataset...")
df = pd.read_json(INPUT_PATH, lines=True)
print(f"Total rows loaded: {len(df)}")

Loading raw dataset...
Total rows loaded: 276139


In [9]:
# Filter for Cardiology and Cardiothoracic Surgery, Vascular Surgery
df_filtered = df[df["specialty"].isin(["Cardiology", "Cardiothoracic Surgery", "Vascular Surgery"])]
print(f"Filtered (Cardiology + Cardiothoracic Surgery, Vascular Surgery): {len(df_filtered)}")

# Sample 5k from each to balance
sample_size = 5000
cardiology_df = df_filtered[df_filtered["specialty"] == "Cardiology"].sample(n=sample_size, random_state=42)
cardiothoracic_df = df_filtered[df_filtered["specialty"] == "Cardiothoracic Surgery"].sample(n=2000, random_state=42)
vascular_df = df_filtered[df_filtered["specialty"] == "Vascular Surgery"].sample(n=sample_size, random_state=42)

# Combine and Shuffle
balanced_df = pd.concat([cardiology_df, cardiothoracic_df, vascular_df], ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final balanced dataset size: {len(balanced_df)}")

Filtered (Cardiology + Cardiothoracic Surgery, Vascular Surgery): 276139
Final balanced dataset size: 12000


In [10]:
# Format for T5 HyDE Training
# We map Question -> Answer (The "Answer" acts as our Hypothetical Document)

print("Formatting data for T5...")

# T5 expects specific column names for some scripts, but we'll use 'input_text' and 'target_text'
# Prefix: "hypothetical answer: " tells the model what task we are doing
df_formatted = pd.DataFrame({
    "input_text": "hypothetical answer: " + balanced_df["question"].astype(str),
    "target_text": balanced_df["answer"].astype(str)
})

# Preview
print("\nSample Training Pair:")
print("INPUT :", df_formatted.iloc[0]["input_text"])
print("TARGET:", df_formatted.iloc[0]["target_text"][:200] + "...") # Truncate for display

Formatting data for T5...

Sample Training Pair:
INPUT : hypothetical answer: How do vasodilating blockers reduce the risk of all-cause mortality in heart failure patients?

TARGET: Vasodilating blockers have been shown to reduce the risk of all-cause mortality in heart failure patients by up to 65% after just six months of treatment. These blockers work by dilating blood vessels...


In [12]:
# Split and Save
# 90% Training, 10% Validation
train_df, val_df = train_test_split(df_formatted, test_size=0.10, random_state=42, shuffle=True)

print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")

# Save to JSONL
train_df.to_json(OUTPUT_TRAIN, orient="records", lines=True)
val_df.to_json(OUTPUT_VAL, orient="records", lines=True)

print(f"\nSaved files:\n1. {OUTPUT_TRAIN}\n2. {OUTPUT_VAL}")
print("Ready for fine-tuning!")


Training set size: 10800
Validation set size: 1200

Saved files:
1. t5_hyde_train_split.jsonl
2. t5_hyde_val_split.jsonl
Ready for fine-tuning!
